# Exploring Corpus

This notebook explores and describes the data that is included in the formed corpus.

- What is the variation in header names?
- What is the frequency of words in sentences (group Parkinson vs non-Parkinson)?

## Initialisation

In [ ]:
# meta
__author__ ="Jennefer Beenen"
__version__ = "1.0"
__email__ = "j.beenen@pl.hanze.nl"
__status__ = "Development"
# __date__ = "2025-05-06"

In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# https://amueller.github.io/word_cloud/
from wordcloud import WordCloud
from wordcloud import STOPWORDS

#### Settings

In [ ]:
# load data
# filename = 'df_xml2corpus_by_sentence_pmid_concept_3600_250425'
filename = 'df_xml2corpus_by_sentence_pmid_free_3600_250606'
df = pd.read_csv(f'../data/corpus/{filename}.csv')
# print info
df.info()

In [ ]:
# function
# exploration
def unique_values_per_column(df: pd.DataFrame):
    """Prints the unique values found in each column of a data frame."""
    for column in df.columns:
        if df[column].nunique() < 25:
            print(f"'{column}' ({df[column].nunique()}):\n{df[column].unique()}\n")
        else:
            print(f"'{column}' ({df[column].nunique()}): (only first 25 values are shown)\n{df[column].unique()[:25]}\n")

In [ ]:
# plot wordcloud
# def get_parkinson_sentences(serie):
#     """Create wordcloud around Parkinson's Disease"""
#     # Replace "PD" with " parkinson disease ".
#     serie = serie.str.replace("PD", "parkinson's disease", case= True)

#     # Make all sentence_text lower case & remove puntuations
#     # https://www.geeksforgeeks.org/python/generating-word-cloud-python/
#     serie = serie.str.lower().str.strip('.?!')

#     # Search for sentences that include 'Parkinson' (or/and "PD"?)
#     sentences = serie.loc[serie.str.findall("parkinson's disease").str.len() > 0].values

#     # Report number of sentences found
#     print(f"{len(sentences)} sentences were found for 'parkinson's disease' (incl. PD, which was replaced by 'parkinson's disease').")

#     return sentences

def create_wordcloud(sentences: list):
    """Create WordCloud object, 16:9 ratio, top 100 words"""
    # Create WordCloud object, 16:9 ratio, top 100 words
    wordcloud = WordCloud(
        width=1600,
        height=900,
        max_words= 100,
        stopwords= STOPWORDS.update(['one', 'two', 'three', 'first', 'study']),
        colormap= 'PiYG'
    ).generate(" ".join(sentences)) # join words to form one long 'sentence'

    # Plotting
    plt.imshow(wordcloud)
    plt.axis('off')
    plt.show()

In [ ]:
def get_sentences(serie, keyword: str) -> list:
    """"""
    # Make all sentence_text lower case & remove puntuations
    # https://www.geeksforgeeks.org/python/generating-word-cloud-python/
    serie = serie.str.lower().str.strip('.?!')
    # Search for sentences that include `keyword`
    sentences = serie.loc[serie.str.findall(keyword).str.len() > 0].values

    # Report number of sentences found
    print(f"{len(sentences)} sentences were found for '{keyword}'.")

    # (Ready to create wordcloud)
    return sentences

### Replace 'PD' with "parkinson's disease"

I'm expecting Parkinson's Disease to be often abbrevated to PD.

In [ ]:
# Current state:
df.head()

In [ ]:
# Check if "PD" can mean Parkinson.
PD_indexes = df['sentence_text'].str.findall("PD").str.len() > 0
# Show
df.loc[PD_indexes, 'sentence_text'].values

As shown above. PD often used as an abbreviation for parkinson's disease.
(also see https://www.ncbi.nlm.nih.gov/books/NBK536722/)

In [ ]:
# Replace " PD " with " parkinson disease ".
df['sentence_text'] = df['sentence_text'].str.replace("PD", "parkinson's disease", case= True)
# Show if correction was successful 
df.loc[PD_indexes, 'sentence_text'].values

In [ ]:
# Make all sentence_text lower case & remove puntuations
# https://www.geeksforgeeks.org/python/generating-word-cloud-python/
df['sentence_text'] = df['sentence_text'].str.lower().str.strip('.?!')

In [ ]:
# Search for sentences that include 'Parkinson'
sentences = df.loc[df['sentence_text'].str.findall("parkinson's disease").str.len() > 0, 'sentence_text'].values
sentences

Note that total number of sentences has now increased: 

3843: when searching only on "PD".           

4154: now together with "parkinson's disease".

## Frequency of header names

In [ ]:
# empty sentence_text
df[df['sentence_text'].isna()]

In [ ]:
# explore unique values
unique_values_per_column(df)

In [ ]:
# turn all head_name s to lower case
df['head_name'] = df['head_name'].str.lower()

In [ ]:
# head
df.head(20)

In [ ]:
# plot the distribution of the 'head_name' column
df.groupby(['head_name'])['paper_id'].nunique().sort_values(ascending=False).head(20).plot(kind='bar', figsize=(10, 6))

# annotate value above each bar
for i, v in enumerate(df.groupby(['head_name'])['paper_id'].nunique().sort_values(ascending=False).head(20)):
    plt.text(i, v + 0.5, str(v), ha='center', va='bottom')

plt.title('Distribution of head_name')
plt.ylabel('Number of unique paper_id')
# plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Frequency words (wordcloud)

Wordcloud python library works the best with English words (for auto removing stopwords) and removes "'s" from text. See [documentation](https://amueller.github.io/word_cloud/_modules/wordcloud/wordcloud.html#WordCloud).

**NOTE:** The processed papers are already biased to (pesticides AND parkinson's disease).

### Example

In [ ]:
# Example from Michiel Noback

# Choose text myself:
# I Want To Live - Borislav Slavov (Baldur's Gate 3)
words = """
I feel your breath upon my neck
A soft caress as cold as death (Cold as death)
I didn't know you well back then
I blame it all on luck and vain (Luck and vain)
Your blood like wine, I wanted in
Oh darling, get me drunk and make me feel

It's not my fault
I'm not to blame
These ain't my sins
I broke my chains
There's more to do
And I still want to live

I feel your breath, upon my neck
A soft caress, as cold as death (Cold as death)
I feel your heart-beat in my soul
Our futures bound, our bodies know (Bodies know)
Your blood like wine, I wanted in
Oh darling get me drunk, invite me in

It's not my fault
I'm not to blame
These ain't my sins
I broke my chains
There's more to do
If I can only live

I can't go yet
Don't let me die
I'll never stop
Until I'm done
But just tonight
Maybe I'll rest in peace

I feel your breath upon my neck
A soft caress as cold as death (Cold as death)
I hear your heart-beat in my soul
Our endings bound, our bodies know

I can't go yet
Don't let me die
I want to live
My only one
There's more to do, if we can only live
The clock won't stop and this is what we get
"""

wordcloud = WordCloud(width=600, height=400).generate(words)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

### Parkinson's disease

#### Explaining `get_parkinson_sentences()` & `create_wordcloud()`

In [ ]:
# # Split sentences to words, flatten into a single string, and remove stopwords
# stopwords = set(STOPWORDS)
# stopwords.update(['one', 'two', 'three', 'study'])

# wordcloud_ready = [word for sent in np.char.split(sentences.astype("U"), " ") for word in sent if word not in stopwords]
# # NOTE: Next time we can use `stopwords` in `WordCloud` for passing stopwords.. (And I used one for-loop too many)

In [ ]:
# # Create WordCloud object, 16:9 ratio,, top 50 words
# wordcloud = WordCloud(
#     width=1600,
#     height=900,
#     max_words= 100,
#     # stopwords=
#     colormap= 'PiYG'
# ).generate(" ".join(wordcloud_ready))

# # Plotting
# plt.imshow(wordcloud)
# plt.axis('off')
# plt.show()

#### Apply function

In [ ]:
create_wordcloud(
    get_sentences(df["sentence_text"], "parkinson's disease")
)

### Pesticides

In [ ]:
# Search for sentences that include 'Pesticide'
create_wordcloud(
    get_sentences(df["sentence_text"], "pesticides")
)

In [ ]:
# Search for sentences that include 'Pesticide'
create_wordcloud(
    get_sentences(df["sentence_text"], "paraquat")
)

## Explore Michiel Nobacks wordcloud code

In [ ]:
# Imports
from nltk.corpus import stopwords

In [ ]:
# Tokenise words
stops = set(stopwords.words('english'))
stops.update(['one', 'two', 'three', 'first', 'study'])

In [ ]:
# What is the difference between `STOPWORDS` from Wordcloud and `stopwords` from nltk?
print(f"{len(stops.symmetric_difference(STOPWORDS))} are symmetric differing between `STOPWORDS` from Wordcloud and `stopwords` from nltk.")